In [43]:
!pip install pyodbc


In [44]:
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd
import requests
from sqlalchemy import create_engine
import os
from datetime import datetime

# --- LOGGING FUNCTION (Taaki save_logs error na de) ---
def save_logs(message):
    log_dir = './log'
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    with open('./log/code_log.txt', 'a') as f:
        f.write(f"{datetime.now()}: {message}\n")

# --- EXTRACT PHASE ---
soup = BeautifulSoup(requests.get('https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks').text,'html.parser')
table = soup.find('span',string='By total assets').find_next('table')
df_main = pd.read_html(StringIO(str(table)))[0]
save_logs("Data has been extracted")
(df_main)

## Transform
import pandas as pd

def Transform(df, csv_path):
    # 1. Read CSV and convert to dict. 
    exchange_rate = pd.read_csv(csv_path, index_col=0).to_dict()['Rate']
    save_logs("Data has been Transformed")
    
    # 2. Match exact casing.
    rates_lower = {k.lower(): v for k, v in exchange_rate.items()}
    
    pkr_rate = rates_lower.get('pkr')
    if pkr_rate is None:
        raise KeyError("Could not find 'PKR' or 'Pkr' in the CSV file's index.")
        
    # --- CRITICAL FIX 1: Pehle Wikipedia ke text assets ko number mein badla ---
    df.columns = [c.strip() for c in df.columns]
    usd_col = 'Total assets (2022) (US$ billion)'
    df['USD_NUMERIC'] = df[usd_col].astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)

    # 3. Calculate and multiply the columns directly
    df['Market cap (PKr billion)'] = df[usd_col] * exchange_rate['PKR']
    df['Market cap (INR billion)'] = df[usd_col] * exchange_rate['INR']
    df['Market cap (EuR billion)'] = df[usd_col] * exchange_rate['EUR']
    df['Market cap (GBP billion)'] = df[usd_col] * exchange_rate['GBP']
    
    # --- CRITICAL FIX 2: Variable names ko theek kiya taaki overwrite na hon ---
    df['total_asset_clean'] = df['USD_NUMERIC'].round(2)
    df['pkr_clean'] = (df['USD_NUMERIC'] * exchange_rate['PKR']).round(2)
    df['inr_clean'] = (df['USD_NUMERIC'] * exchange_rate['INR']).round(2) 
    df['eur_clean'] = (df['USD_NUMERIC'] * exchange_rate['EUR']).round(2)
    df['gbp_clean'] = (df['USD_NUMERIC'] * exchange_rate['GBP']).round(2)

    # --- CRITICAL FIX 3: Wikipedia ke real column names use kiye ---
    df_final = pd.DataFrame({
        'Rank': df['Rank'],
        'Bank_name': df['Bank name'],
        'total_asset': df['total_asset_clean'],
        'Market_cap_pkr': df['pkr_clean'],
        'Market_cap_inr': df['inr_clean'],
        'Market_cap_eur': df['eur_clean'],   
        'Market_cap_gbp': df['gbp_clean']    
    })
    
    save_logs("Data has been Transformed")
    return df_final

# --- CRITICAL FIX 4: Function ko data pass karke execution variable save kiya ---
df_transformed = Transform(df_main, './INPUT/Book1.csv')

def load_data(df, output_csv_path, server_name, database_name, table_name):
    try:
        # Step A: CSV mein data save karna
        folder = os.path.dirname(output_csv_path)
        if folder and not os.path.exists(folder):
            os.makedirs(folder)
        df.to_csv(output_csv_path, index=False)
        save_logs('Data save in to csv')
        print(f"Success: CSV file created at {output_csv_path}")
        
        # Step B: MSSQL Database mein data load karna
        save_logs("LOAD PHASE (MSSQL) STARTED")

        connection_url = (
            f"mssql+pyodbc://@{server_name}/{database_name}"
            f"?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
        )
        engine = create_engine(connection_url)

        # --- CRITICAL FIX 5: String table_name pass kiya aur variable error theek kiya ---
        df.to_sql(table_name, con=engine, if_exists='append', index=False)

        save_logs(f"Load Phase Completed. Data successfully saved to table: {table_name}")
        print(f"Success: Data successfully loaded into MSSQL table '{table_name}'.")
        
    except Exception as e:
        save_logs(f"Load Phase Failed. Error: {str(e)}")
        print(f"Error in Load Phase: {e}")

# --- PIPELINE RUN ---
SERVER = 'ZAPC40' 
DATABASE = 'Banking_Etl_pipeline'
TABLE = 'bank_Etl'
OUTPUT_CSV = './OUTPUT/cleaned_data.csv'

# --- CRITICAL FIX 6: Transformed table pass ki pipeline chalane ke liye ---
load_data(df_transformed, OUTPUT_CSV, SERVER, DATABASE, TABLE)


Success: CSV file created at ./OUTPUT/cleaned_data.csv
Success: Data successfully loaded into MSSQL table 'bank_Etl'.


In [46]:
if __name__ == '__main__':
    url = 'https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks'
    csv_path = './input/Book1.csv'
    output_csv_path = './OUTPUT/cleaned_data.csv'
    
    df = extract(url)

    df = Transform(df,csv_path)
    load_data(df_transformed, OUTPUT_CSV, SERVER, DATABASE, TABLE)

    df

https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks
    Rank                                        Bank name  \
0      1  Industrial and Commercial Bank of China Limited   
1      2                          China Construction Bank   
2      3                       Agricultural Bank of China   
3      4                                    Bank of China   
4      5                                   JPMorgan Chase   
..   ...                                              ...   
95    96                          National Bank of Canada   
96    97                                  Macquarie Group   
97    98                      Nationwide Building Society   
98    99                                Raiffeisen Gruppe   
99   100                             First Abu Dhabi Bank   

    Total assets (2022) (US$ billion)  
0                             5742.86  
1                             5016.81  
2                             4919.03  
3           